In [1]:
from torch.utils.data import DataLoader
import torch
from sklearn.utils.class_weight import compute_class_weight
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup,AutoTokenizer,DistilBertModel
from torch.nn.modules.loss import BCEWithLogitsLoss,CrossEntropyLoss
import numpy as np
from torchvision.models import resnet18,ResNet18_Weights
from torchvision import transforms
from modules import CollateFunction,CreationDataset,Train,DistilbertResnetModel
import sys
import os
current_dir = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(current_dir), "..")))
from common_files import split,creation_dataframe

In [2]:
#Creation of the dataframes from the jsonl files
original_train_df=creation_dataframe("../data/train.jsonl")
val_df=creation_dataframe("../data/dev.jsonl")
original_train_index_list=list(original_train_df.index)

In [3]:
train_df,test_df=split(original_train_df)

In [4]:
#Creation of the datasets
train_dataset=CreationDataset(train_df,"../CLIP_model/modules/clip_embeddings/train_clip_embeddings.pt")
val_dataset=CreationDataset(val_df,"../CLIP_model/modules/clip_embeddings/val_clip_embeddings.pt")

In [5]:
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [6]:
collate_object=CollateFunction(tokenizer)

In [7]:
#Creation of the dataloaders
batch_size=32
train_dataloader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True,collate_fn=collate_object.collate_fn,drop_last=True)
val_dataloader=DataLoader(val_dataset,batch_size=batch_size,shuffle=True,collate_fn=collate_object.collate_fn,drop_last=True)

In [8]:
#Use of the resnet18 model initialized with its default pretrained weights as the Vision model
resnet_model=resnet18(weights=ResNet18_Weights.DEFAULT)

In [9]:
#Use of the pretrained distilbert model as the transformer model
distilbert_model=DistilBertModel.from_pretrained("distilbert-base-uncased")

In [10]:
#Use of the class weights to compensate imabalances of the dataset and make more accurate predictions
class_weight=compute_class_weight("balanced",classes=np.unique(train_df["label"]),y=train_df["label"].to_numpy())
class_weight=torch.tensor(class_weight,dtype=torch.float32)
print(class_weight)

tensor([0.7798, 1.3934])


In [12]:
#Training of the model using only concat interactions and clip embeddings
model=DistilbertResnetModel(distilbert_model,resnet_model,with_clip_image=False,with_clip_text=False,concat_interaction=True,dropout=0.3,fc_layer_sizes=[384])
n_epochs=10
n_steps=len(train_dataloader)*n_epochs
n_warmup_steps=int(0.1*n_steps)
loss_fn=CrossEntropyLoss(weight=class_weight)
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
trainer=Train(model=model,loss_fn=loss_fn,n_epochs=n_epochs,device=device,n_steps=n_steps,n_warmup_steps=n_warmup_steps,n_frozen_distilbert_layers=6,n_frozen_resnet_layers=4,weight_decay=5e-4,lr=0.01,with_clip_images=False,with_clip_text=False,concat=True)
trainer.run_training(train_dataloader=train_dataloader,val_dataloader=val_dataloader,path="./modules/train_savings/interaction_emb_concatenation_only")

2026-03-18 20:05:33.856 | INFO     | modules.train:run_training:186 - Epoch 0 :
2026-03-18 20:33:50.318 | INFO     | modules.train:run_training:276 - Epoch 0: Train Loss = 0.7255863437592733
2026-03-18 20:33:50.318 | INFO     | modules.train:run_training:277 - Epoch 0: Train Accuracy = 0.45672071129707115
2026-03-18 20:33:50.318 | INFO     | modules.train:run_training:278 - Epoch 0: Train F1 = 0.44830941301476945
2026-03-18 20:33:50.318 | INFO     | modules.train:run_training:280 - Epoch 0: Validation Loss = 0.6753092010815939
2026-03-18 20:33:50.318 | INFO     | modules.train:run_training:281 - Epoch 0: Validation Accuracy = 0.5020833333333333
2026-03-18 20:33:50.318 | INFO     | modules.train:run_training:282 - Epoch 0: Validation F1 = 0.4632356573297255
2026-03-18 20:33:51.603 | INFO     | modules.train:run_training:186 - Epoch 1 :
2026-03-18 20:59:05.636 | INFO     | modules.train:run_training:276 - Epoch 1: Train Loss = 0.6471512408685485
2026-03-18 20:59:05.636 | INFO     | modul

In [11]:
#Training of the model using concat interactions and clip embeddings
model=DistilbertResnetModel(distilbert_model,resnet_model,with_clip_image=True,with_clip_text=True,concat_interaction=True,dropout=0.3,fc_layer_sizes=[640])
n_epochs=11
n_steps=len(train_dataloader)*n_epochs
n_warmup_steps=int(0.1*n_steps)
loss_fn=CrossEntropyLoss(weight=class_weight)
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
trainer=Train(model=model,loss_fn=loss_fn,n_epochs=n_epochs,device=device,n_steps=n_steps,n_warmup_steps=n_warmup_steps,n_frozen_distilbert_layers=6,n_frozen_resnet_layers=4,weight_decay=5e-4,lr=0.01,with_clip_images=True,with_clip_text=True,concat=True)
trainer.run_training(train_dataloader=train_dataloader,val_dataloader=val_dataloader,path="./modules/train_savings/interaction_emb_concatenation_with_clip")

2026-03-18 17:01:16.223 | INFO     | modules.train:run_training:186 - Epoch 0 :
2026-03-18 17:23:27.745 | INFO     | modules.train:run_training:276 - Epoch 0: Train Loss = 0.7148757180409452
2026-03-18 17:23:27.745 | INFO     | modules.train:run_training:277 - Epoch 0: Train Accuracy = 0.5092834728033473
2026-03-18 17:23:27.745 | INFO     | modules.train:run_training:278 - Epoch 0: Train F1 = 0.517328070092139
2026-03-18 17:23:27.745 | INFO     | modules.train:run_training:280 - Epoch 0: Validation Loss = 0.6988885164260864
2026-03-18 17:23:27.745 | INFO     | modules.train:run_training:281 - Epoch 0: Validation Accuracy = 0.5166666666666667
2026-03-18 17:23:27.745 | INFO     | modules.train:run_training:282 - Epoch 0: Validation F1 = 0.5121865746175817
2026-03-18 17:23:29.160 | INFO     | modules.train:run_training:186 - Epoch 1 :
2026-03-18 17:58:30.644 | INFO     | modules.train:run_training:276 - Epoch 1: Train Loss = 0.6526838378926202
2026-03-18 17:58:30.763 | INFO     | modules.

In [13]:
#Training of the model using mean combination only
model=DistilbertResnetModel(distilbert_model,resnet_model,with_clip_image=False,with_clip_text=False,concat_interaction=False,dropout=0.3,fc_layer_sizes=[384])
n_epochs=11
n_steps=len(train_dataloader)*n_epochs
n_warmup_steps=int(0.1*n_steps)
loss_fn=CrossEntropyLoss(weight=class_weight)
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
trainer=Train(model=model,loss_fn=loss_fn,n_epochs=n_epochs,device=device,n_steps=n_steps,n_warmup_steps=n_warmup_steps,n_frozen_distilbert_layers=6,n_frozen_resnet_layers=4,weight_decay=5e-4,lr=0.01,with_clip_images=False,with_clip_text=False,concat=False)
trainer.run_training(train_dataloader=train_dataloader,val_dataloader=val_dataloader,path="./modules/train_savings/mean_emb_combination_only")

2026-03-18 22:00:07.006 | INFO     | modules.train:run_training:186 - Epoch 0 :
2026-03-18 22:35:18.176 | INFO     | modules.train:run_training:276 - Epoch 0: Train Loss = 0.8069721739661244
2026-03-18 22:35:18.176 | INFO     | modules.train:run_training:277 - Epoch 0: Train Accuracy = 0.6244769874476988
2026-03-18 22:35:18.193 | INFO     | modules.train:run_training:278 - Epoch 0: Train F1 = 0.5142158402439275
2026-03-18 22:35:18.193 | INFO     | modules.train:run_training:280 - Epoch 0: Validation Loss = 0.8828246037165324
2026-03-18 22:35:18.193 | INFO     | modules.train:run_training:281 - Epoch 0: Validation Accuracy = 0.49583333333333335
2026-03-18 22:35:18.193 | INFO     | modules.train:run_training:282 - Epoch 0: Validation F1 = 0.3300951717734447
2026-03-18 22:35:19.516 | INFO     | modules.train:run_training:186 - Epoch 1 :
2026-03-18 23:07:51.983 | INFO     | modules.train:run_training:276 - Epoch 1: Train Loss = 0.6669293829087932
2026-03-18 23:07:51.990 | INFO     | module

In [14]:
#Training of the model using mean combination with CLIP embeddings

model=DistilbertResnetModel(distilbert_model,resnet_model,with_clip_image=True,with_clip_text=True,concat_interaction=False,dropout=0.3,fc_layer_sizes=[640])
n_epochs=11
n_steps=len(train_dataloader)*n_epochs
n_warmup_steps=int(0.1*n_steps)
loss_fn=CrossEntropyLoss(weight=class_weight)
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
trainer=Train(model=model,loss_fn=loss_fn,n_epochs=n_epochs,device=device,n_steps=n_steps,n_warmup_steps=n_warmup_steps,n_frozen_distilbert_layers=6,n_frozen_resnet_layers=4,weight_decay=5e-4,lr=0.01,with_clip_images=True,with_clip_text=True,concat=False)
trainer.run_training(train_dataloader=train_dataloader,val_dataloader=val_dataloader,path="./modules/train_savings/mean_emb_combination_with_clip")

2026-03-19 01:08:28.996 | INFO     | modules.train:run_training:186 - Epoch 0 :
2026-03-19 01:29:23.700 | INFO     | modules.train:run_training:276 - Epoch 0: Train Loss = 0.7305291103019874
2026-03-19 01:29:23.700 | INFO     | modules.train:run_training:277 - Epoch 0: Train Accuracy = 0.5673378661087866
2026-03-19 01:29:23.700 | INFO     | modules.train:run_training:278 - Epoch 0: Train F1 = 0.5440970207223675
2026-03-19 01:29:23.700 | INFO     | modules.train:run_training:280 - Epoch 0: Validation Loss = 0.7410974264144897
2026-03-19 01:29:23.700 | INFO     | modules.train:run_training:281 - Epoch 0: Validation Accuracy = 0.5229166666666667
2026-03-19 01:29:23.700 | INFO     | modules.train:run_training:282 - Epoch 0: Validation F1 = 0.4455821388153822
2026-03-19 01:29:24.150 | INFO     | modules.train:run_training:186 - Epoch 1 :
2026-03-19 01:50:37.556 | INFO     | modules.train:run_training:276 - Epoch 1: Train Loss = 0.6444030936542415
2026-03-19 01:50:37.558 | INFO     | modules

In [13]:
model=DistilbertResnetModel(distilbert_model,resnet_model,with_clip_image=True,with_clip_text=True,concat_interaction=True,dropout=0.3,fc_layer_sizes=[640])
model_parameters=torch.load("modules/train_savings/interaction_emb_concatenation_with_clip/model_state.pt")
model.load_state_dict(model_parameters)

<All keys matched successfully>